In [4]:
import pandas as pd

In [5]:
# Path to the template TSV file containing the headers
template_file_path = "node_template/submission_case_template.tsv"

# Read the template TSV file to extract the headers
df_template = pd.read_csv(template_file_path, sep="\t", nrows=0)  # Read only the header
headers = df_template.columns.tolist()  # Extract the headers as a list

In [6]:
# Observational Patients

# File path to the input CSV file
file_path_obs = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/OBS Final Datasets/OBS_SUBJECTS.csv"

# Read the file using pandas
df_obs_input = pd.read_csv(file_path_obs, sep=",", dtype=str)
# 2025.5.23 Exclude the invalid patient with usubjid == "31014"
df_obs_input = df_obs_input[df_obs_input["usubjid"] != "31014"]

# Initialize a DataFrame with the defined headers and same index as df_obs_input
df_obs_output = pd.DataFrame(index=df_obs_input.index, columns=headers)

# Assign fixed values using .loc to align with the index
df_obs_output.loc[:, "*type"] = "case"
df_obs_output.loc[:, "project_id"] = "ARDaC-AlcHepNet"
df_obs_output.loc[:, "*studies.submitter_id"] = "obs"
df_obs_output.loc[:, "index_date"] = "Study Enrollment"

# Map dynamic values using .apply
df_obs_output["*submitter_id"] = df_obs_input["usubjid"].apply(lambda x: f"{x}_obs" if pd.notna(x) else None)
df_obs_output["cohort"] = df_obs_input["obs_arm"].apply(lambda x: x.split(":")[-1].strip() if pd.notna(x) else None)
df_obs_output["study_site"] = df_obs_input["site"].apply(lambda x: x.strip() if pd.notna(x) else None)
df_obs_output["vital_status"] = df_obs_input["ALIVE"].apply(lambda x: "alive" if x == "Y" else "dead" if x == "N" else None)

# Display the resulting DataFrame
print("Observational Patients Output DataFrame preview:")
print(df_obs_output.head())

Observational Patients Output DataFrame preview:
  *type       project_id *submitter_id *studies.submitter_id actarm ah_hosp  \
0  case  ARDaC-AlcHepNet     11001_obs                   obs    NaN     NaN   
1  case  ARDaC-AlcHepNet     11002_obs                   obs    NaN     NaN   
2  case  ARDaC-AlcHepNet     11003_obs                   obs    NaN     NaN   
3  case  ARDaC-AlcHepNet     11004_obs                   obs    NaN     NaN   
4  case  ARDaC-AlcHepNet     11006_obs                   obs    NaN     NaN   

  ah_hosp_num aki_status bari_surgery                                  cohort  \
0         NaN        NaN          NaN                           Healthy donor   
1         NaN        NaN          NaN                           Healthy donor   
2         NaN        NaN          NaN  Heavy drinker with alcoholic hepatitis   
3         NaN        NaN          NaN                           Healthy donor   
4         NaN        NaN          NaN  Heavy drinker with alcoholic hep

In [7]:
# 2025.05.22 V2.0.2 Add AKI to OBS patient
file_path_aki_obs = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/OBS Final Datasets/OBS_ABX_AKI_CX.csv"
df_aki_obs = pd.read_csv(file_path_aki_obs, sep=",", dtype=str)

# Ensure 'akiaernyn' has a default of 'Unknown' for missing values
df_aki_obs["akiaernyn"] = df_aki_obs["akiaernyn"].fillna("Unknown")

# Build a mapping from usubjid to akiaernyn
aki_map_obs = df_aki_obs.set_index("usubjid")["akiaernyn"].to_dict()

# Assign aki_status: use map if present, else default to 'Unknown'
df_obs_output["aki_status"] = df_obs_input["usubjid"].apply(lambda x: aki_map_obs.get(x, "Unknown"))

In [8]:
# Clinical Trial Patients
# File path to the RCT_SUBJECTS.csv file
file_path_rct = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/RCT Final Datasets/RCT_SUBJECTS.csv"

# Read the RCT_SUBJECTS.csv file
df_input_rct = pd.read_csv(file_path_rct, sep=",", dtype=str)

# Initialize a new DataFrame with the defined headers and the same index as df_input_rct
df_rct_output = pd.DataFrame(index=df_input_rct.index, columns=headers)

# Assign fixed values using .loc for proper alignment
df_rct_output.loc[:, "*type"] = "case"
df_rct_output.loc[:, "project_id"] = "ARDaC-AlcHepNet"
df_rct_output.loc[:, "*studies.submitter_id"] = "clinical"
df_rct_output.loc[:, "index_date"] = "Study Enrollment"

# Map dynamic values based on the input file
df_rct_output["*submitter_id"] = df_input_rct["usubjid"].apply(lambda x: f"{x}_clinical" if pd.notna(x) else None)
df_rct_output["actarm"] = df_input_rct["rct_arm"].apply(lambda x: x.strip() if pd.notna(x) else None)
df_rct_output["rct_meld_strata"] = df_input_rct["rct_meld_strata"].apply(lambda x: x.strip() if pd.notna(x) else None)
df_rct_output["study_site"] = df_input_rct["site"].apply(lambda x: x.strip() if pd.notna(x) else None)
df_rct_output["vital_status"] = df_input_rct["ALIVE"].apply(lambda x: "alive" if x == "Y" else "dead" if x == "N" else None)

# Fill other unmapped columns with NaN for consistency
for col in df_rct_output.columns:
    if col not in ["*type", "project_id", "*submitter_id", "*studies.submitter_id", "actarm", 
                   "rct_meld_strata", "study_site", "vital_status", "index_date"]:
        df_rct_output[col] = None

# Display the first few rows of the output DataFrame for verification
print("Output RCT DataFrame preview:")
print(df_rct_output.head())

Output RCT DataFrame preview:
  *type       project_id   *submitter_id *studies.submitter_id  \
0  case  ARDaC-AlcHepNet  11055_clinical              clinical   
1  case  ARDaC-AlcHepNet  11058_clinical              clinical   
2  case  ARDaC-AlcHepNet  11066_clinical              clinical   
3  case  ARDaC-AlcHepNet  11067_clinical              clinical   
4  case  ARDaC-AlcHepNet  11071_clinical              clinical   

            actarm ah_hosp ah_hosp_num aki_status bari_surgery cohort  ...  \
0       Prednisone    None        None       None         None   None  ...   
1       Prednisone    None        None       None         None   None  ...   
2       Prednisone    None        None       None         None   None  ...   
3  Anakinra + Zinc    None        None       None         None   None  ...   
4       Prednisone    None        None       None         None   None  ...   

  days_90_aki days_90_survival days_to_aki days_to_consent days_to_death  \
0        None             No

In [9]:
# 2025.05.22 V2.0.2 Add AKI to RCT patient
file_path_ae = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/RCT Final Datasets/RCT_ADVERSE_EVENTS.csv"
df_ae = pd.read_csv(file_path_ae, sep=",", dtype=str)

# Filter patients with ae_aki_indicator == "1"
aki_patients = df_ae[df_ae["ae_aki_indicator"] == "1"]["usubjid"].unique()

# Assign 'yes' to patients with AE=1, and 'no' to all others
df_rct_output["aki_status"] = df_input_rct["usubjid"].apply(lambda x: "Yes" if x in aki_patients else "No")

In [10]:
# Export the "CASE" node.

# Save the Combined DataFrame to TSV
rct_output_path = "case_rct_DCC_data_release_v2-0-2.tsv"
obs_output_path = "case_obs_DCC_data_release_v2-0-2.tsv"

df_rct_output.to_csv(rct_output_path, sep="\t", index=False, header=True)
print(f"RCT patients file saved as: {rct_output_path}")
df_obs_output.to_csv(obs_output_path, sep="\t", index=False, header=True)
print(f"Observational patients file saved as: {obs_output_path}")

RCT patients file saved as: case_rct_DCC_data_release_v2-0-2.tsv
Observational patients file saved as: case_obs_DCC_data_release_v2-0-2.tsv
